In [ ]:
import pandas as pd
import numpy as np
import datetime as dt


pce = pd.read_csv('data/pce.csv')
shifted = np.zeros(len(pce.index))
shifted[:-1] = pce.PCECTPI[1:]
#returns = np.zeros(len(pce.index))
#returns[:-1] = -(pce.PCECTPI[:-1].to_numpy()-pce.PCECTPI[1:].to_numpy())/pce.PCECTPI[:-1].to_numpy()

pce['Close'] = shifted
pce['Open'] = pce.PCECTPI

pce.observation_date = pce.observation_date.apply(lambda x: dt.datetime.strptime(x, "%Y-%m-%d"))
pce = pce.set_index('observation_date')
pce = pce[['Close', 'Open']]
pce = pce.iloc[:-1]
pce

def load_pce():
    pce = pd.read_csv('data/pce.csv')
    shifted = np.zeros(len(pce.index))
    shifted[:-1] = pce.PCECTPI[1:]
    #returns = np.zeros(len(pce.index))
    #returns[:-1] = -(pce.PCECTPI[:-1].to_numpy()-pce.PCECTPI[1:].to_numpy())/pce.PCECTPI[:-1].to_numpy()

    pce['Close'] = shifted
    pce['Open'] = pce.PCECTPI

    pce.observation_date = pce.observation_date.apply(lambda x: dt.datetime.strptime(x, "%Y-%m-%d"))
    pce = pce.set_index('observation_date')
    pce = pce[['Close', 'Open']]
    pce = pce.iloc[:-1]

    return pce

,Close,Open
observation_date,,
2000-01-01,73.568,73.219
2000-04-01,74.042,73.568
2000-07-01,74.460,74.042
2000-10-01,75.012,74.460
2001-01-01,75.363,75.012
...,...,...
2024-07-01,124.703,123.930
2024-10-01,125.759,124.703
2025-01-01,126.424,125.759


In [68]:
from utils.data import *

In [69]:


start = dt.datetime(year=2000, month=1, day=1)
end = dt.datetime(year=2023, month=1, day=1)

aapl = get_hist('AAPL', start=start, end=end).reset_index()
aapl.Date = aapl.Date.apply(lambda x: x.replace(tzinfo=None))
aapl = aapl.set_index("Date")

stock_covar(pce, aapl)
#aapl.index

array([[2.37906186e-05, 1.55264446e-04],
       [1.55264446e-04, 4.63223293e-02]])

In [93]:
def add_covariates_to_covar(Sigma, all_stocks, covars:list):
    n1 = Sigma.shape[0] 
    n2 = n1 + len(covars)
    new_Sigma = np.zeros((n2, n2))

    new_Sigma[:n1, :n1] = Sigma
    for j, c in enumerate(covars):
        c_idx1 = j + n1

        for i, t1 in enumerate(all_stocks):
            hist1 = get_hist(t1, start, end).reset_index()
            hist1.Date = hist1.Date.apply(lambda x: x.replace(tzinfo=None))
            hist1 = hist1.set_index("Date")

            cov = stock_covar(hist1, c)


            new_Sigma[i, c_idx1] = cov[0, 1]
            new_Sigma[c_idx1, i] = cov[1, 0]
            new_Sigma[c_idx1, c_idx1] = cov[1, 1]

        for j2, c2 in enumerate(covars):
            if j2 > j:
                c_idx2 = j2 + n1
                cov2 = stock_covar(c, c2)

                new_Sigma[c_idx1, c_idx1] = cov2[0, 0]
                new_Sigma[c_idx2, c_idx1] = cov2[0, 1]
                new_Sigma[c_idx1, c_idx2] = cov2[1, 0]
                new_Sigma[c_idx2, c_idx2] = cov2[1, 1]
                                                 
    Sxx = Sigma
    Sxy = new_Sigma[:n1, n1:]
    Syx = Sxy.T
    Syy = new_Sigma[n1:, n1:]

    return new_Sigma, (Sxx, Sxy, Syx, Syy)

def add_covariates_to_mu(mu, covars:list):
    n1 = mu.shape[0] 
    n2 = n1 + len(covars)
    new_mu = np.zeros((n2))

    new_mu[:n1] = mu
    for j, c in enumerate(covars):
        c_idx1 = j + n1

        new_mu[c_idx1] = expected_returns(c)
    return new_mu, (mu, new_mu[n1:])

In [ ]:
from utils.data import *

pce = load_pce()
effr = load_ffr()

all_stocks = ["NVDA", "AAPL", "BA", 'DB']
start = dt.datetime(year=2000, month=1, day=1)
end = dt.datetime(year=2023, month=1, day=1)


Sigma = generate_Sigma(all_stocks, start, end)
mu = generate_mu(all_stocks, start, end)

new_sigma, sep_Sigma = add_covariates_to_covar(Sigma, all_stocks, [pce, effr], start, end)
new_mu, sep_mu = add_covariates_to_mu(mu, [pce, effr])

(array([ 0.10712951,  0.07541541,  0.0325801 , -0.00268711]),
 array([0.00546368, 0.72538535]))

In [4]:
def conditional_moments(mu, Sigma, a):
    (Sxx, Sxy, Syx, Syy) = Sigma
    (mu_x, mu_y) = mu
    mu_a = mu_x + Sxy @ np.linalg.inv(Syy)@(a-mu_y)
    Sigma_a = Sxx - Sxy@np.linalg.inv(Syy)@Syx

    return mu_a, Sigma_a

In [9]:
conditional_moments(sep_mu, sep_Sigma, [0.005, 4])

(array([ 0.11784797,  0.06861307,  0.03435628, -0.01046164]),
 array([[0.11562611, 0.03790047, 0.01591687, 0.0303147 ],
        [0.03790047, 0.04527312, 0.00398775, 0.01509983],
        [0.01591687, 0.00398775, 0.02974967, 0.02073798],
        [0.0303147 , 0.01509983, 0.02073798, 0.03747639]]))

In [ ]:
def load_data(filename):
    df = pd.read_csv(filename)
    shifted = np.zeros(len(pce.index))
    shifted[:-1] = pce.PCECTPI[1:]
    #returns = np.zeros(len(pce.index))
    #returns[:-1] = -(pce.PCECTPI[:-1].to_numpy()-pce.PCECTPI[1:].to_numpy())/pce.PCECTPI[:-1].to_numpy()

    df['Close'] = shifted
    df['Open'] = pce.PCECTPI

    df.observation_date = pce.observation_date.apply(lambda x: dt.datetime.strptime(x, "%Y-%m-%d"))
    df = pce.set_index('observation_date')
    df = pce[['Close', 'Open']]
    df = pce.iloc[:-1]

    return pce

In [ ]:
irx = get_hist("^IRX", start, end).reset_index()
irx["observation_date"] = irx.Date.apply(lambda x: x.replace(tzinfo=None))
irx = irx.set_index("observation_date")
irx = irx[['Close', 'Open']]

irx.to_csv("data/effr.csv")